<a href="https://colab.research.google.com/github/ROHANIRL/ML-FUEL_DELAY_PROJECT/blob/main/ML_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.makedirs('data', exist_ok=True)

with open("data_prep.py", 'w') as f:
    f.write("\"\"\"\ndata_prep.py\n------------\nLoads the ship fuel dataset. If the real Kaggle CSV\n(ship_fuel_efficiency.csv) is present in ./data/, it's used directly.\nOtherwise a synthetic dataset matching the same schema is generated,\nso the pipeline can be tested before the real file is available.\n\"\"\"\n\nimport os\nimport numpy as np\nimport pandas as pd\n\nDATA_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), \"data\", \"ship_fuel_efficiency.csv\")\n\nSHIP_TYPES = [\"Oil Service Boat\", \"Fishing Trawler\", \"Surfer Boat\", \"Tanker Ship\"]\nFUEL_TYPES = [\"Diesel\", \"HFO\"]\nWEATHER = [\"Calm\", \"Moderate\", \"Stormy\"]\n\n# The one genuinely external assumption in this pipeline: nominal\n# (scheduled) service speed per vessel class, in knots.\nNOMINAL_SPEED_KNOTS = {\n    \"Oil Service Boat\": 10.0,\n    \"Fishing Trawler\": 8.0,\n    \"Surfer Boat\": 20.0,\n    \"Tanker Ship\": 14.0,\n}\n\nMONTH_ORDER = {\n    \"January\": 1, \"February\": 2, \"March\": 3, \"April\": 4, \"May\": 5, \"June\": 6,\n    \"July\": 7, \"August\": 8, \"September\": 9, \"October\": 10, \"November\": 11, \"December\": 12,\n}\n\n\ndef _generate_synthetic(n_ships=120, voyages_per_ship=12, seed=42) -> pd.DataFrame:\n    \"\"\"Generates a placeholder dataset matching the real schema, for testing\n    before the real Kaggle CSV is available.\"\"\"\n    rng = np.random.default_rng(seed)\n    rows = []\n    months = list(MONTH_ORDER.keys())\n    for ship_idx in range(n_ships):\n        ship_id = f\"NG{ship_idx:03d}\"\n        ship_type = rng.choice(SHIP_TYPES)\n        nominal_speed = NOMINAL_SPEED_KNOTS[ship_type]\n        fuel_type = rng.choice(FUEL_TYPES, p=[0.55, 0.45])\n        for v in range(voyages_per_ship):\n            distance = float(np.clip(rng.normal(220, 110), 20, 499))\n            weather = rng.choice(WEATHER, p=[0.5, 0.35, 0.15])\n            weather_drag = {\"Calm\": 1.00, \"Moderate\": 1.08, \"Stormy\": 1.22}[weather]\n            engine_efficiency = float(np.clip(rng.normal(85, 6), 70, 95))\n            true_speed = max(3.0, rng.normal(nominal_speed, nominal_speed * 0.12))\n            base_k = {\"Oil Service Boat\": 0.23, \"Fishing Trawler\": 0.28,\n                      \"Surfer Boat\": 0.035, \"Tanker Ship\": 0.35}[ship_type]\n            efficiency_penalty = (100 - engine_efficiency) / 100 * 0.6 + 1.0\n            fuel_consumption = (base_k * (true_speed ** 2) * distance * weather_drag\n                                 * efficiency_penalty * rng.normal(1.0, 0.04))\n            fuel_consumption = float(np.clip(fuel_consumption, 237.88, 24648.52))\n            co2_factor = 2.68 if fuel_type == \"Diesel\" else 3.11\n            co2_emissions = fuel_consumption * co2_factor * rng.normal(1.0, 0.03)\n            rows.append({\n                \"ship_id\": ship_id, \"ship_type\": ship_type,\n                \"route_id\": f\"R{rng.integers(1, 12)}\", \"month\": months[v % 12],\n                \"distance\": round(distance, 2), \"fuel_type\": fuel_type,\n                \"fuel_consumption\": round(fuel_consumption, 2),\n                \"CO2_emissions\": round(co2_emissions, 2), \"weather_conditions\": weather,\n                \"engine_efficiency\": round(engine_efficiency, 2),\n            })\n    return pd.DataFrame(rows)\n\n\ndef load_data() -> tuple[pd.DataFrame, bool]:\n    \"\"\"Returns (dataframe, is_real_data). is_real_data=False means the\n    synthetic fallback was used because data/ship_fuel_efficiency.csv\n    was not found.\"\"\"\n    if os.path.exists(DATA_PATH):\n        df = pd.read_csv(DATA_PATH)\n        if \"date\" not in df.columns and \"month\" in df.columns:\n            month_num = df[\"month\"].map(MONTH_ORDER)\n            df[\"date\"] = pd.to_datetime({\"year\": 2023, \"month\": month_num, \"day\": 1})\n        elif \"date\" in df.columns:\n            df[\"date\"] = pd.to_datetime(df[\"date\"])\n        return df, True\n    else:\n        return _generate_synthetic(), False\n\n\nif __name__ == \"__main__\":\n    df, is_real = load_data()\n    print(f\"Loaded {'REAL' if is_real else 'SYNTHETIC (placeholder)'} data: {df.shape}\")\n    print(df.head())\n")

with open("feature_engineering.py", 'w') as f:
    f.write("\"\"\"\nfeature_engineering.py\n-----------------------\nBuilds the features that differentiate this project from a standard\n\"predict fuel consumption\" pipeline:\n\n1. implied_speed_knots  -- backed out of fuel_consumption using the\n   Admiralty relation (fuel per distance ~ speed^2), calibrated PER\n   SHIP TYPE from the dataset's own medians.\n\n2. delay_hours          -- schedule deviation relative to nominal\n   service speed. Positive = late, negative = ahead of schedule.\n\n3. Economic layer        -- bunker_cost_usd, demurrage_cost_usd,\n   total_voyage_cost_usd.\n\nASSUMPTIONS (stated explicitly, not hidden):\n  - distance is assumed to be in nautical miles\n  - nominal service speed per ship type (data_prep.NOMINAL_SPEED_KNOTS)\n    is external, not derived from data\n  - fuel prices and demurrage rates below are indicative figures\n\"\"\"\n\nimport numpy as np\nimport pandas as pd\n\nfrom data_prep import NOMINAL_SPEED_KNOTS\n\nFUEL_PRICE_USD_PER_LITRE = {\"HFO\": 0.42, \"Diesel\": 0.72}\n\nDEMURRAGE_USD_PER_HOUR = {\n    \"Oil Service Boat\": 45, \"Fishing Trawler\": 20,\n    \"Surfer Boat\": 35, \"Tanker Ship\": 120,\n}\n\nADMIRALTY_EXPONENT = 2.0\n\n\ndef add_implied_speed(df: pd.DataFrame) -> pd.DataFrame:\n    df = df.copy()\n    k_by_type = {}\n    for ship_type, nominal_speed in NOMINAL_SPEED_KNOTS.items():\n        subset = df[df[\"ship_type\"] == ship_type]\n        if len(subset) == 0:\n            continue\n        med_fuel = subset[\"fuel_consumption\"].median()\n        med_dist = subset[\"distance\"].median()\n        k_by_type[ship_type] = med_fuel / (nominal_speed ** ADMIRALTY_EXPONENT * med_dist)\n    df[\"_k\"] = df[\"ship_type\"].map(k_by_type)\n    df[\"implied_speed_knots\"] = np.sqrt(df[\"fuel_consumption\"] / (df[\"_k\"] * df[\"distance\"]))\n    df.drop(columns=[\"_k\"], inplace=True)\n    return df\n\n\ndef add_delay_variable(df: pd.DataFrame) -> pd.DataFrame:\n    df = df.copy()\n    df[\"nominal_speed_knots\"] = df[\"ship_type\"].map(NOMINAL_SPEED_KNOTS)\n    df[\"actual_time_hours\"] = df[\"distance\"] / df[\"implied_speed_knots\"]\n    df[\"expected_time_hours\"] = df[\"distance\"] / df[\"nominal_speed_knots\"]\n    df[\"delay_hours\"] = df[\"actual_time_hours\"] - df[\"expected_time_hours\"]\n    return df\n\n\ndef add_economic_layer(df: pd.DataFrame) -> pd.DataFrame:\n    df = df.copy()\n    df[\"fuel_price_usd_per_litre\"] = df[\"fuel_type\"].map(FUEL_PRICE_USD_PER_LITRE)\n    df[\"bunker_cost_usd\"] = df[\"fuel_consumption\"] * df[\"fuel_price_usd_per_litre\"]\n    df[\"demurrage_rate_usd_per_hour\"] = df[\"ship_type\"].map(DEMURRAGE_USD_PER_HOUR)\n    df[\"demurrage_cost_usd\"] = df[\"delay_hours\"].clip(lower=0) * df[\"demurrage_rate_usd_per_hour\"]\n    df[\"total_voyage_cost_usd\"] = df[\"bunker_cost_usd\"] + df[\"demurrage_cost_usd\"]\n    return df\n\n\ndef engineer_all(df: pd.DataFrame) -> pd.DataFrame:\n    df = add_implied_speed(df)\n    df = add_delay_variable(df)\n    df = add_economic_layer(df)\n    return df\n\n\nif __name__ == \"__main__\":\n    from data_prep import load_data\n\n    raw, is_real = load_data()\n    feats = engineer_all(raw)\n    cols = [\"ship_type\", \"distance\", \"fuel_consumption\", \"implied_speed_knots\",\n            \"delay_hours\", \"total_voyage_cost_usd\"]\n    print(feats[cols].describe())\n")

with open("models.py", 'w') as f:
    f.write("\"\"\"\nmodels.py\n---------\nThree models, each answering a different question:\n\n1. Polynomial Regression  -> interpretable baseline, consistent with\n   the assumed quadratic Admiralty relation.\n2. Random Forest           -> main tabular model; handles categorical\n   interactions and gives feature importance.\n3. LSTM                    -> trained on PER-VESSEL VOYAGE SEQUENCES\n   (ordered by date) to capture fuel-consumption drift across a\n   vessel's operating history -- a pattern the other two structurally\n   cannot see, since they treat each voyage independently.\n\"\"\"\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, r2_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler\nfrom sklearn.linear_model import LinearRegression\n\nNUMERIC_FEATURES = [\"distance\", \"engine_efficiency\"]\nCATEGORICAL_FEATURES = [\"ship_type\", \"fuel_type\", \"weather_conditions\"]\nTARGET = \"fuel_consumption\"\n\n\ndef _split(df, target=TARGET, test_size=0.2, seed=42):\n    X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]\n    y = df[target]\n    return train_test_split(X, y, test_size=test_size, random_state=seed)\n\n\ndef train_polynomial_regression(df, degree=2):\n    X_train, X_test, y_train, y_test = _split(df)\n    preprocess = ColumnTransformer([\n        (\"num\", Pipeline([\n            (\"poly\", PolynomialFeatures(degree=degree, include_bias=False)),\n            (\"scale\", StandardScaler()),\n        ]), NUMERIC_FEATURES),\n        (\"cat\", OneHotEncoder(handle_unknown=\"ignore\"), CATEGORICAL_FEATURES),\n    ])\n    pipe = Pipeline([(\"prep\", preprocess), (\"model\", LinearRegression())])\n    pipe.fit(X_train, y_train)\n    preds = pipe.predict(X_test)\n    metrics = {\"model\": f\"Polynomial Regression (deg={degree})\",\n               \"MAE\": mean_absolute_error(y_test, preds), \"R2\": r2_score(y_test, preds)}\n    return pipe, metrics\n\n\ndef train_random_forest(df, n_estimators=300, max_depth=12):\n    X_train, X_test, y_train, y_test = _split(df)\n    preprocess = ColumnTransformer([\n        (\"num\", \"passthrough\", NUMERIC_FEATURES),\n        (\"cat\", OneHotEncoder(handle_unknown=\"ignore\"), CATEGORICAL_FEATURES),\n    ])\n    pipe = Pipeline([\n        (\"prep\", preprocess),\n        (\"model\", RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth,\n                                          random_state=42, n_jobs=-1)),\n    ])\n    pipe.fit(X_train, y_train)\n    preds = pipe.predict(X_test)\n    metrics = {\"model\": \"Random Forest\", \"MAE\": mean_absolute_error(y_test, preds),\n               \"R2\": r2_score(y_test, preds)}\n\n    feature_names = (NUMERIC_FEATURES\n                      + list(pipe.named_steps[\"prep\"].named_transformers_[\"cat\"]\n                             .get_feature_names_out(CATEGORICAL_FEATURES)))\n    importances = pipe.named_steps[\"model\"].feature_importances_\n    importance_df = pd.DataFrame({\"feature\": feature_names, \"importance\": importances}\n                                  ).sort_values(\"importance\", ascending=False)\n    return pipe, metrics, importance_df\n\n\ndef check_lstm_feasibility(df, min_voyages=8):\n    counts = df.groupby(\"ship_id\").size()\n    feasible = (counts >= min_voyages).sum() >= max(5, 0.5 * counts.shape[0])\n    return feasible, counts\n\n\ndef build_lstm_sequences(df, window=5, target=\"fuel_consumption\"):\n    feature_cols = [\"fuel_consumption\", \"distance\", \"engine_efficiency\"]\n    X_seqs, y_seqs = [], []\n    date_col = \"date\" if \"date\" in df.columns else None\n    sorted_df = df.sort_values(date_col) if date_col else df\n    for ship_id, group in sorted_df.groupby(\"ship_id\"):\n        vals = group[feature_cols].values\n        if len(vals) <= window:\n            continue\n        for i in range(len(vals) - window):\n            X_seqs.append(vals[i:i + window])\n            y_seqs.append(vals[i + window][0])\n    return np.array(X_seqs), np.array(y_seqs)\n\n\ndef train_lstm(df, window=5, epochs=30):\n    from sklearn.preprocessing import StandardScaler\n    from tensorflow import keras\n    from tensorflow.keras import layers\n\n    X, y = build_lstm_sequences(df, window=window)\n    if len(X) < 40:\n        raise ValueError(f\"Only {len(X)} sequences available -- not enough per-vessel \"\n                          \"voyage history for a meaningful LSTM.\")\n\n    n_samples, n_steps, n_feat = X.shape\n    X_flat = X.reshape(-1, n_feat)\n    scaler_X = StandardScaler().fit(X_flat)\n    X_scaled = scaler_X.transform(X_flat).reshape(n_samples, n_steps, n_feat)\n    scaler_y = StandardScaler().fit(y.reshape(-1, 1))\n    y_scaled = scaler_y.transform(y.reshape(-1, 1)).ravel()\n\n    split = int(0.8 * n_samples)\n    X_train, X_test = X_scaled[:split], X_scaled[split:]\n    y_train, y_test = y_scaled[:split], y_scaled[split:]\n\n    model = keras.Sequential([\n        layers.Input(shape=(n_steps, n_feat)),\n        layers.LSTM(32, return_sequences=False),\n        layers.Dense(16, activation=\"relu\"),\n        layers.Dense(1),\n    ])\n    model.compile(optimizer=\"adam\", loss=\"mse\")\n    model.fit(X_train, y_train, epochs=epochs, batch_size=16, verbose=0, validation_split=0.15)\n\n    preds_scaled = model.predict(X_test, verbose=0).ravel()\n    preds = scaler_y.inverse_transform(preds_scaled.reshape(-1, 1)).ravel()\n    y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()\n\n    metrics = {\"model\": f\"LSTM (window={window})\", \"MAE\": mean_absolute_error(y_true, preds),\n               \"R2\": r2_score(y_true, preds), \"n_sequences\": len(X)}\n    return model, (scaler_X, scaler_y), metrics\n\n\nif __name__ == \"__main__\":\n    from data_prep import load_data\n    from feature_engineering import engineer_all\n\n    raw, is_real = load_data()\n    df = engineer_all(raw)\n    print(f\"Data source: {'REAL' if is_real else 'SYNTHETIC'}\")\n\n    _, poly_metrics = train_polynomial_regression(df)\n    print(poly_metrics)\n\n    _, rf_metrics, importance_df = train_random_forest(df)\n    print(rf_metrics)\n    print(importance_df.head(8).to_string(index=False))\n\n    feasible, counts = check_lstm_feasibility(df)\n    print(f\"LSTM feasible: {feasible} (voyages/ship median: {counts.median()})\")\n    if feasible:\n        try:\n            _, _, lstm_metrics = train_lstm(df)\n            print(lstm_metrics)\n        except ValueError as e:\n            print(f\"LSTM skipped: {e}\")\n")

with open("robustness_check.py", 'w') as f:
    f.write("\"\"\"\nrobustness_check.py\n--------------------\nAnswers \"how do you know your model isn't overfitting, and where does\nit go wrong?\" Three checks on the Random Forest model:\n  1. Train vs Test R2/MAE (overfitting check)\n  2. 5-fold cross-validation (stability check)\n  3. Error breakdown by ship_type and weather_conditions\n\"\"\"\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, r2_score\nfrom sklearn.model_selection import train_test_split, KFold, cross_val_score\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder\n\nNUMERIC_FEATURES = [\"distance\", \"engine_efficiency\"]\nCATEGORICAL_FEATURES = [\"ship_type\", \"fuel_type\", \"weather_conditions\"]\nTARGET = \"fuel_consumption\"\n\n\ndef _build_pipeline():\n    preprocess = ColumnTransformer([\n        (\"num\", \"passthrough\", NUMERIC_FEATURES),\n        (\"cat\", OneHotEncoder(handle_unknown=\"ignore\"), CATEGORICAL_FEATURES),\n    ])\n    return Pipeline([(\"prep\", preprocess),\n                      (\"model\", RandomForestRegressor(n_estimators=300, max_depth=12,\n                                                        random_state=42, n_jobs=-1))])\n\n\ndef check_overfitting(df):\n    X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]\n    y = df[TARGET]\n    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n    pipe = _build_pipeline()\n    pipe.fit(X_train, y_train)\n    train_r2 = r2_score(y_train, pipe.predict(X_train))\n    test_r2 = r2_score(y_test, pipe.predict(X_test))\n    gap = train_r2 - test_r2\n    verdict = (\"LOW overfitting risk.\" if gap < 0.05 else\n               \"MODERATE gap -- some overfitting.\" if gap < 0.15 else\n               \"HIGH gap -- likely overfitting.\")\n    print(f\"Train R2: {train_r2:.4f} | Test R2: {test_r2:.4f} | Gap: {gap:.4f} | {verdict}\")\n    return {\"train_r2\": train_r2, \"test_r2\": test_r2, \"gap\": gap, \"verdict\": verdict}\n\n\ndef check_cross_validation(df, k=5):\n    X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]\n    y = df[TARGET]\n    pipe = _build_pipeline()\n    kf = KFold(n_splits=k, shuffle=True, random_state=42)\n    scores = cross_val_score(pipe, X, y, cv=kf, scoring=\"r2\")\n    print(f\"{k}-fold CV -- Mean R2: {scores.mean():.4f}, Std: {scores.std():.4f}\")\n    return {\"fold_scores\": scores.tolist(), \"mean\": scores.mean(), \"std\": scores.std()}\n\n\ndef error_breakdown(df):\n    X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]\n    y = df[TARGET]\n    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n    pipe = _build_pipeline()\n    pipe.fit(X_train, y_train)\n    preds = pipe.predict(X_test)\n    results = X_test.copy()\n    results[\"actual\"] = y_test.values\n    results[\"predicted\"] = preds\n    results[\"pct_error\"] = 100 * np.abs(results[\"actual\"] - results[\"predicted\"]) / results[\"actual\"]\n    print(\"\\nBy ship type:\")\n    print(results.groupby(\"ship_type\")[\"pct_error\"].mean().round(1).sort_values(ascending=False).to_string())\n    print(\"\\nBy weather:\")\n    print(results.groupby(\"weather_conditions\")[\"pct_error\"].mean().round(1).sort_values(ascending=False).to_string())\n    return results\n\n\ndef run(df):\n    print(\"Running robustness checks on Random Forest...\\n\")\n    overfit = check_overfitting(df)\n    cv = check_cross_validation(df)\n    errors = error_breakdown(df)\n    return overfit, cv, errors\n\n\nif __name__ == \"__main__\":\n    from data_prep import load_data\n    from feature_engineering import engineer_all\n\n    raw, is_real = load_data()\n    df = engineer_all(raw)\n    print(f\"Data source: {'REAL' if is_real else 'SYNTHETIC'}\\n\")\n    run(df)\n")

with open("model_tuning.py", 'w') as f:
    f.write("\"\"\"\nmodel_tuning.py\n----------------\n1. HYPERPARAMETER TUNING via RandomizedSearchCV, instead of guessed defaults.\n2. UNCERTAINTY ESTIMATES using the spread of predictions across individual\n   trees in the forest, so a recommendation can report a range, not just\n   a bare point estimate.\n\"\"\"\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, r2_score\nfrom sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder\n\nNUMERIC_FEATURES = [\"distance\", \"engine_efficiency\"]\nCATEGORICAL_FEATURES = [\"ship_type\", \"fuel_type\", \"weather_conditions\"]\nTARGET = \"fuel_consumption\"\n\n\ndef tune_random_forest(df, n_iter=30, seed=42):\n    X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]\n    y = df[TARGET]\n    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)\n\n    preprocess = ColumnTransformer([\n        (\"num\", \"passthrough\", NUMERIC_FEATURES),\n        (\"cat\", OneHotEncoder(handle_unknown=\"ignore\"), CATEGORICAL_FEATURES),\n    ])\n    pipe = Pipeline([(\"prep\", preprocess), (\"model\", RandomForestRegressor(random_state=seed, n_jobs=-1))])\n\n    param_dist = {\n        \"model__n_estimators\": [100, 200, 300, 500, 700],\n        \"model__max_depth\": [6, 8, 10, 12, 16, 20, None],\n        \"model__min_samples_split\": [2, 4, 6, 10],\n        \"model__min_samples_leaf\": [1, 2, 4, 6],\n        \"model__max_features\": [\"sqrt\", \"log2\", 0.5, 0.7, 1.0],\n    }\n\n    kf = KFold(n_splits=5, shuffle=True, random_state=seed)\n    search = RandomizedSearchCV(pipe, param_distributions=param_dist, n_iter=n_iter, cv=kf,\n                                  scoring=\"r2\", random_state=seed, n_jobs=-1)\n    search.fit(X_train, y_train)\n\n    best_pipe = search.best_estimator_\n    test_pred = best_pipe.predict(X_test)\n    test_r2 = r2_score(y_test, test_pred)\n    test_mae = mean_absolute_error(y_test, test_pred)\n\n    baseline_pipe = Pipeline([(\"prep\", preprocess),\n                               (\"model\", RandomForestRegressor(n_estimators=300, max_depth=12,\n                                                                 random_state=seed, n_jobs=-1))])\n    baseline_pipe.fit(X_train, y_train)\n    baseline_pred = baseline_pipe.predict(X_test)\n    baseline_r2 = r2_score(y_test, baseline_pred)\n\n    print(f\"Best params: {search.best_params_}\")\n    print(f\"Tuned Test R2: {test_r2:.4f} | Baseline Test R2: {baseline_r2:.4f} | \"\n          f\"Improvement: {test_r2 - baseline_r2:+.4f}\")\n\n    return {\"best_pipe\": best_pipe, \"best_params\": search.best_params_,\n            \"tuned_test_r2\": test_r2, \"tuned_test_mae\": test_mae,\n            \"baseline_test_r2\": baseline_r2, \"improvement\": test_r2 - baseline_r2}\n\n\ndef predict_with_uncertainty(pipe, X_row):\n    \"\"\"Returns a 90% prediction interval using the spread across the\n    forest's individual trees, instead of a single point estimate.\"\"\"\n    preprocess = pipe.named_steps[\"prep\"]\n    model = pipe.named_steps[\"model\"]\n    X_transformed = preprocess.transform(X_row)\n    tree_preds = np.array([tree.predict(X_transformed)[0] for tree in model.estimators_])\n    return {\n        \"point_estimate\": tree_preds.mean(), \"std\": tree_preds.std(),\n        \"lower_90\": np.percentile(tree_preds, 5), \"upper_90\": np.percentile(tree_preds, 95),\n    }\n\n\nif __name__ == \"__main__\":\n    from data_prep import load_data\n    from feature_engineering import engineer_all\n\n    raw, is_real = load_data()\n    df = engineer_all(raw)\n    print(f\"Data source: {'REAL' if is_real else 'SYNTHETIC'}\\n\")\n\n    result = tune_random_forest(df, n_iter=30)\n\n    example = pd.DataFrame([{\"distance\": 250, \"engine_efficiency\": 85,\n                              \"ship_type\": \"Tanker Ship\", \"fuel_type\": \"HFO\",\n                              \"weather_conditions\": \"Moderate\"}])\n    unc = predict_with_uncertainty(result[\"best_pipe\"], example)\n    print(f\"\\nPoint estimate: {unc['point_estimate']:.1f} litres \"\n          f\"(90% interval: [{unc['lower_90']:.1f}, {unc['upper_90']:.1f}])\")\n")

with open("pareto.py", 'w') as f:
    f.write("\"\"\"\npareto.py\n---------\nThe actual decision-support layer: sweeps candidate speeds for a given\nvoyage scenario and finds the cost-minimizing speed. Also supports a\ndelay-tolerance CONSTRAINT, so a customer's time expectation can be\npriced in rather than ignored.\n\"\"\"\n\nimport numpy as np\nimport pandas as pd\n\nfrom data_prep import NOMINAL_SPEED_KNOTS\nfrom feature_engineering import FUEL_PRICE_USD_PER_LITRE, DEMURRAGE_USD_PER_HOUR\n\n\ndef _calibrate_k(df, ship_type, admiralty_exponent=2.0):\n    subset = df[df[\"ship_type\"] == ship_type]\n    nominal_speed = NOMINAL_SPEED_KNOTS[ship_type]\n    med_fuel = subset[\"fuel_consumption\"].median()\n    med_dist = subset[\"distance\"].median()\n    return med_fuel / (nominal_speed ** admiralty_exponent * med_dist)\n\n\ndef speed_cost_curve(df, ship_type, distance, fuel_type=\"HFO\",\n                      weather=\"Moderate\", speed_range=None, admiralty_exponent=2.0):\n    if speed_range is None:\n        nominal = NOMINAL_SPEED_KNOTS[ship_type]\n        speed_range = np.linspace(nominal * 0.25, nominal * 1.5, 80)\n\n    weather_drag = {\"Calm\": 1.00, \"Moderate\": 1.08, \"Stormy\": 1.22}[weather]\n    k = _calibrate_k(df, ship_type, admiralty_exponent)\n    nominal_speed = NOMINAL_SPEED_KNOTS[ship_type]\n    fuel_price = FUEL_PRICE_USD_PER_LITRE[fuel_type]\n    demurrage_rate = DEMURRAGE_USD_PER_HOUR[ship_type]\n\n    rows = []\n    for speed in speed_range:\n        fuel = k * (speed ** admiralty_exponent) * distance * weather_drag\n        actual_time = distance / speed\n        expected_time = distance / nominal_speed\n        delay = max(0.0, actual_time - expected_time)\n        bunker_cost = fuel * fuel_price\n        demurrage_cost = delay * demurrage_rate\n        rows.append({\n            \"speed_knots\": speed, \"fuel_consumption_l\": fuel, \"delay_hours\": delay,\n            \"bunker_cost_usd\": bunker_cost, \"demurrage_cost_usd\": demurrage_cost,\n            \"total_cost_usd\": bunker_cost + demurrage_cost,\n        })\n    return pd.DataFrame(rows)\n\n\ndef recommend_speed(df, ship_type, distance, fuel_type=\"HFO\", weather=\"Moderate\"):\n    \"\"\"Pure cost minimization, ignoring lateness entirely.\"\"\"\n    curve = speed_cost_curve(df, ship_type, distance, fuel_type, weather)\n    best = curve.loc[curve[\"total_cost_usd\"].idxmin()]\n    nominal_speed = NOMINAL_SPEED_KNOTS[ship_type]\n    nominal_row = curve.iloc[(curve[\"speed_knots\"] - nominal_speed).abs().idxmin()]\n    savings_usd = nominal_row[\"total_cost_usd\"] - best[\"total_cost_usd\"]\n    savings_pct = 100 * savings_usd / nominal_row[\"total_cost_usd\"]\n    comparison = {\n        \"nominal_speed_knots\": nominal_row[\"speed_knots\"],\n        \"nominal_total_cost_usd\": nominal_row[\"total_cost_usd\"],\n        \"optimal_speed_knots\": best[\"speed_knots\"],\n        \"optimal_total_cost_usd\": best[\"total_cost_usd\"],\n        \"savings_usd\": savings_usd, \"savings_pct\": savings_pct,\n    }\n    return best, curve, comparison\n\n\ndef recommend_speed_with_max_delay(df, ship_type, distance, max_delay_hours,\n                                     fuel_type=\"HFO\", weather=\"Moderate\"):\n    \"\"\"\n    Finds the cheapest speed subject to a HARD CAP on delay -- this is\n    how a customer's time expectation gets priced in. max_delay_hours=0\n    forces on-time-or-early arrival, even if that costs more fuel.\n    \"\"\"\n    curve = speed_cost_curve(df, ship_type, distance, fuel_type, weather)\n    feasible = curve[curve[\"delay_hours\"] <= max_delay_hours]\n    if len(feasible) == 0:\n        best_constrained = curve.loc[curve[\"delay_hours\"].idxmin()]\n        feasible_found = False\n    else:\n        best_constrained = feasible.loc[feasible[\"total_cost_usd\"].idxmin()]\n        feasible_found = True\n\n    unconstrained_best = curve.loc[curve[\"total_cost_usd\"].idxmin()]\n    extra_cost = best_constrained[\"total_cost_usd\"] - unconstrained_best[\"total_cost_usd\"]\n    extra_cost_pct = (100 * extra_cost / unconstrained_best[\"total_cost_usd\"]\n                       if unconstrained_best[\"total_cost_usd\"] > 0 else 0)\n\n    result = {\n        \"max_delay_hours\": max_delay_hours, \"feasible_found\": feasible_found,\n        \"constrained_speed_knots\": best_constrained[\"speed_knots\"],\n        \"constrained_delay_hours\": best_constrained[\"delay_hours\"],\n        \"constrained_total_cost_usd\": best_constrained[\"total_cost_usd\"],\n        \"unconstrained_speed_knots\": unconstrained_best[\"speed_knots\"],\n        \"unconstrained_delay_hours\": unconstrained_best[\"delay_hours\"],\n        \"unconstrained_total_cost_usd\": unconstrained_best[\"total_cost_usd\"],\n        \"extra_cost_usd\": extra_cost, \"extra_cost_pct\": extra_cost_pct,\n    }\n    return result, curve\n\n\ndef delay_tolerance_tradeoff(df, ship_type, distance, fuel_type=\"HFO\", weather=\"Moderate\",\n                               max_delay_options=None):\n    \"\"\"Shows how total cost changes as the acceptable delay is tightened --\n    directly answers 'how much does respecting the customer's schedule cost us?'\"\"\"\n    if max_delay_options is None:\n        unconstrained_curve = speed_cost_curve(df, ship_type, distance, fuel_type, weather)\n        max_possible_delay = unconstrained_curve[\"delay_hours\"].max()\n        max_delay_options = np.linspace(0, max_possible_delay, 8)\n\n    rows = []\n    for max_delay in max_delay_options:\n        result, _ = recommend_speed_with_max_delay(df, ship_type, distance, max_delay, fuel_type, weather)\n        rows.append({\n            \"max_delay_allowed_hours\": max_delay,\n            \"speed_knots\": result[\"constrained_speed_knots\"],\n            \"actual_delay_hours\": result[\"constrained_delay_hours\"],\n            \"total_cost_usd\": result[\"constrained_total_cost_usd\"],\n            \"extra_cost_vs_unconstrained_pct\": result[\"extra_cost_pct\"],\n        })\n    return pd.DataFrame(rows)\n\n\nif __name__ == \"__main__\":\n    from data_prep import load_data\n    from feature_engineering import engineer_all\n\n    raw, is_real = load_data()\n    df = engineer_all(raw)\n    print(f\"Data source: {'REAL' if is_real else 'SYNTHETIC'}\\n\")\n\n    for ship_type in NOMINAL_SPEED_KNOTS:\n        best, curve, comparison = recommend_speed(df, ship_type, distance=250)\n        print(f\"{ship_type}: {comparison['nominal_speed_knots']:.1f} kn -> \"\n              f\"{comparison['optimal_speed_knots']:.1f} kn | \"\n              f\"Savings: {comparison['savings_pct']:.1f}%\")\n")

with open("risk_aware.py", 'w') as f:
    f.write("\"\"\"\nrisk_aware.py\n-------------\nNOVELTY ADDITION: connects the Random Forest's per-prediction uncertainty\n(model_tuning.py) to the speed-recommendation engine (pareto.py).\n\nEvery classical ship-speed-optimization model in the literature (and the\nbase paper's own models) treats the fuel prediction as EXACTLY correct and\npicks the single cost-minimizing speed. This module instead asks: \"what if\nthe fuel prediction is wrong by the amount the model itself says it might\nbe wrong by?\" -- and lets you choose a speed that hedges against that,\ninstead of blindly trusting a point estimate.\n\nThis is a genuinely different recommendation logic, not a cosmetic add-on:\na risk-averse operator and a risk-neutral operator can get DIFFERENT\nrecommended speeds out of this function for the exact same voyage.\n\"\"\"\n\nimport numpy as np\nimport pandas as pd\n\nfrom data_prep import NOMINAL_SPEED_KNOTS\nfrom feature_engineering import FUEL_PRICE_USD_PER_LITRE, DEMURRAGE_USD_PER_HOUR\nfrom pareto import speed_cost_curve\nfrom model_tuning import predict_with_uncertainty\n\n\ndef estimate_relative_uncertainty(tuned_pipe, ship_type, distance, fuel_type, weather, engine_efficiency=85):\n    \"\"\"\n    Uses the trained Random Forest's tree-to-tree spread (model_tuning.py)\n    to estimate how uncertain the fuel prediction is for this scenario,\n    as a FRACTION of the predicted value -- e.g. 0.08 means \"typically\n    +/-8% uncertainty around the point estimate.\"\n    \"\"\"\n    scenario = pd.DataFrame([{\n        \"distance\": distance, \"engine_efficiency\": engine_efficiency,\n        \"ship_type\": ship_type, \"fuel_type\": fuel_type, \"weather_conditions\": weather,\n    }])\n    unc = predict_with_uncertainty(tuned_pipe, scenario)\n    if unc[\"point_estimate\"] <= 0:\n        return 0.1  # fallback if something degenerate happens\n    return unc[\"std\"] / unc[\"point_estimate\"]\n\n\ndef speed_cost_curve_with_risk(df, tuned_pipe, ship_type, distance, fuel_type=\"HFO\",\n                                 weather=\"Moderate\", speed_range=None):\n    \"\"\"\n    Same as pareto.speed_cost_curve, but adds a cost RANGE at every speed,\n    propagating the RF model's relative fuel-prediction uncertainty onto\n    the bunker cost component (the part of total cost that comes from the\n    ML-adjacent fuel estimate).\n    \"\"\"\n    curve = speed_cost_curve(df, ship_type, distance, fuel_type, weather, speed_range)\n    rel_unc = estimate_relative_uncertainty(tuned_pipe, ship_type, distance, fuel_type, weather)\n\n    curve = curve.copy()\n    curve[\"bunker_cost_std_usd\"] = curve[\"bunker_cost_usd\"] * rel_unc\n    curve[\"total_cost_lower_usd\"] = curve[\"total_cost_usd\"] - curve[\"bunker_cost_std_usd\"]\n    curve[\"total_cost_upper_usd\"] = curve[\"total_cost_usd\"] + curve[\"bunker_cost_std_usd\"]\n    curve[\"relative_uncertainty\"] = rel_unc\n    return curve\n\n\ndef recommend_speed_risk_aware(df, tuned_pipe, ship_type, distance, fuel_type=\"HFO\",\n                                 weather=\"Moderate\", risk_aversion=1.0):\n    \"\"\"\n    risk_aversion=0   -> identical to naive recommend_speed (ignores uncertainty)\n    risk_aversion=1   -> picks the speed minimizing (point estimate + 1 std) --\n                          i.e. hedges against the model being about as wrong\n                          as it typically is\n    risk_aversion=2   -> more conservative, hedges against a larger miss\n\n    Returns both the naive (risk-neutral) and risk-aware recommendations so\n    you can show how -- and whether -- they differ.\n    \"\"\"\n    curve = speed_cost_curve_with_risk(df, tuned_pipe, ship_type, distance, fuel_type, weather)\n\n    naive_best = curve.loc[curve[\"total_cost_usd\"].idxmin()]\n\n    curve[\"risk_adjusted_cost_usd\"] = curve[\"total_cost_usd\"] + risk_aversion * curve[\"bunker_cost_std_usd\"]\n    risk_aware_best = curve.loc[curve[\"risk_adjusted_cost_usd\"].idxmin()]\n\n    speed_shift = risk_aware_best[\"speed_knots\"] - naive_best[\"speed_knots\"]\n\n    result = {\n        \"relative_uncertainty_pct\": curve[\"relative_uncertainty\"].iloc[0] * 100,\n        \"naive_speed_knots\": naive_best[\"speed_knots\"],\n        \"naive_cost_usd\": naive_best[\"total_cost_usd\"],\n        \"risk_aware_speed_knots\": risk_aware_best[\"speed_knots\"],\n        \"risk_aware_point_cost_usd\": risk_aware_best[\"total_cost_usd\"],\n        \"risk_aware_worst_case_cost_usd\": risk_aware_best[\"total_cost_upper_usd\"],\n        \"speed_shift_knots\": speed_shift,\n        \"interpretation\": (\n            \"Risk-aware recommendation matches the naive one -- uncertainty didn't \"\n            \"change the answer for this scenario.\" if abs(speed_shift) < 0.1 else\n            f\"Risk-aware recommendation shifts speed by {speed_shift:+.1f} kn to hedge \"\n            f\"against a {curve['relative_uncertainty'].iloc[0]*100:.1f}% typical fuel-prediction error.\"\n        ),\n    }\n    return result, curve\n\n\nif __name__ == \"__main__\":\n    from data_prep import load_data\n    from feature_engineering import engineer_all\n    from model_tuning import tune_random_forest\n\n    raw, is_real = load_data()\n    df = engineer_all(raw)\n    print(f\"Data source: {'REAL' if is_real else 'SYNTHETIC'}\\n\")\n\n    print(\"Tuning Random Forest (needed for uncertainty estimates)...\")\n    tuning_result = tune_random_forest(df, n_iter=20)\n    tuned_pipe = tuning_result[\"best_pipe\"]\n\n    print(\"\\n\" + \"=\" * 60)\n    print(\"RISK-AWARE VS NAIVE RECOMMENDATION\")\n    print(\"=\" * 60)\n    for ship_type in NOMINAL_SPEED_KNOTS:\n        result, curve = recommend_speed_risk_aware(df, tuned_pipe, ship_type, distance=250, risk_aversion=1.0)\n        print(f\"\\n{ship_type} (fuel-prediction uncertainty: {result['relative_uncertainty_pct']:.1f}%)\")\n        print(f\"  Naive (risk-neutral):    {result['naive_speed_knots']:.1f} kn -> ${result['naive_cost_usd']:,.0f}\")\n        print(f\"  Risk-aware:              {result['risk_aware_speed_knots']:.1f} kn -> \"\n              f\"${result['risk_aware_point_cost_usd']:,.0f} \"\n              f\"(worst case: ${result['risk_aware_worst_case_cost_usd']:,.0f})\")\n        print(f\"  {result['interpretation']}\")\n")

with open("api_server.py", 'w') as f:
    f.write("\"\"\"\napi_server.py\n--------------\nExposes the LIVE model running in this Colab session as a web API.\n\nThis is what makes the website genuinely DEPEND on Colab: every calculation\nthe website shows is computed by this server, using whatever df and trained\nmodel are actually loaded right now -- not a snapshot, not baked-in values.\nIf this server stops (Colab disconnects, tab closes, cell stops), the\nwebsite's live calculations stop working too. That dependency is the point.\n\"\"\"\n\nfrom flask import Flask, request, jsonify\nfrom flask_cors import CORS\n\nfrom pareto import recommend_speed, recommend_speed_with_max_delay, speed_cost_curve, delay_tolerance_tradeoff\nfrom data_prep import NOMINAL_SPEED_KNOTS\n\n\ndef create_app(df, tuned_pipe=None, is_real=True):\n    \"\"\"\n    df: the live, currently-loaded dataset (engineered features already applied)\n    tuned_pipe: optional -- the tuned Random Forest pipeline, needed for\n                risk-aware recommendations. If None, risk-aware endpoint\n                returns an error explaining it's not available yet.\n    \"\"\"\n    app = Flask(__name__)\n    CORS(app)\n\n    @app.route('/health', methods=['GET'])\n    def health():\n        return jsonify({\n            \"status\": \"alive\",\n            \"data_source\": \"REAL Kaggle dataset\" if is_real else \"SYNTHETIC placeholder\",\n            \"rows_loaded\": len(df),\n            \"risk_aware_available\": tuned_pipe is not None,\n        })\n\n    @app.route('/recommend', methods=['POST'])\n    def recommend():\n        data = request.get_json(force=True)\n        ship_type = data.get('ship_type', 'Tanker Ship')\n        distance = float(data.get('distance', 250))\n        weather = data.get('weather', 'Moderate')\n        fuel_type = data.get('fuel_type', 'HFO')\n        max_delay = float(data.get('max_delay_hours', 10))\n\n        if ship_type not in NOMINAL_SPEED_KNOTS:\n            return jsonify({\"error\": f\"Unknown ship_type: {ship_type}\"}), 400\n\n        best, curve, comparison = recommend_speed(df, ship_type, distance, fuel_type, weather)\n        constrained, _ = recommend_speed_with_max_delay(df, ship_type, distance, max_delay, fuel_type, weather)\n\n        return jsonify({\n            \"data_source\": \"REAL Kaggle dataset\" if is_real else \"SYNTHETIC placeholder\",\n            \"rows_used\": len(df),\n            \"nominal\": {\n                \"speed_knots\": round(comparison[\"nominal_speed_knots\"], 2),\n                \"cost_usd\": round(comparison[\"nominal_total_cost_usd\"], 2),\n            },\n            \"cheapest\": {\n                \"speed_knots\": round(comparison[\"optimal_speed_knots\"], 2),\n                \"cost_usd\": round(comparison[\"optimal_total_cost_usd\"], 2),\n                \"savings_usd\": round(comparison[\"savings_usd\"], 2),\n                \"savings_pct\": round(comparison[\"savings_pct\"], 2),\n            },\n            \"your_choice\": {\n                \"speed_knots\": round(constrained[\"constrained_speed_knots\"], 2),\n                \"cost_usd\": round(constrained[\"constrained_total_cost_usd\"], 2),\n                \"delay_hours\": round(constrained[\"constrained_delay_hours\"], 2),\n            },\n            \"speed_curve\": {\n                \"speed_knots\": curve[\"speed_knots\"].round(2).tolist(),\n                \"total_cost_usd\": curve[\"total_cost_usd\"].round(2).tolist(),\n            },\n        })\n\n    @app.route('/delay_tradeoff', methods=['POST'])\n    def delay_tradeoff():\n        data = request.get_json(force=True)\n        ship_type = data.get('ship_type', 'Tanker Ship')\n        distance = float(data.get('distance', 250))\n        weather = data.get('weather', 'Moderate')\n        fuel_type = data.get('fuel_type', 'HFO')\n\n        tradeoff = delay_tolerance_tradeoff(df, ship_type, distance, fuel_type, weather)\n        return jsonify({\n            \"max_delay_allowed_hours\": tradeoff[\"max_delay_allowed_hours\"].round(2).tolist(),\n            \"total_cost_usd\": tradeoff[\"total_cost_usd\"].round(2).tolist(),\n        })\n\n    @app.route('/risk_aware', methods=['POST'])\n    def risk_aware():\n        if tuned_pipe is None:\n            return jsonify({\"error\": \"Risk-aware model not available -- \"\n                                      \"run Section 5 (hyperparameter tuning) first.\"}), 400\n        from risk_aware import recommend_speed_risk_aware\n\n        data = request.get_json(force=True)\n        ship_type = data.get('ship_type', 'Tanker Ship')\n        distance = float(data.get('distance', 250))\n        weather = data.get('weather', 'Moderate')\n        fuel_type = data.get('fuel_type', 'HFO')\n        risk_aversion = float(data.get('risk_aversion', 1.0))\n\n        result, _ = recommend_speed_risk_aware(df, tuned_pipe, ship_type, distance, fuel_type, weather, risk_aversion)\n        return jsonify({k: (round(v, 2) if isinstance(v, float) else v) for k, v in result.items()})\n\n    return app\n\n\nif __name__ == \"__main__\":\n    # Standalone test using synthetic data -- not how this is normally run\n    # (normally called from the notebook with the live df already loaded).\n    from data_prep import load_data\n    from feature_engineering import engineer_all\n\n    raw, is_real = load_data()\n    df = engineer_all(raw)\n    app = create_app(df, tuned_pipe=None, is_real=is_real)\n    app.run(port=5000, debug=True)\n")

print('All modules written:', ["data_prep.py", "feature_engineering.py", "models.py", "robustness_check.py", "model_tuning.py", "pareto.py", "risk_aware.py", "api_server.py"])

All modules written: ['data_prep.py', 'feature_engineering.py', 'models.py', 'robustness_check.py', 'model_tuning.py', 'pareto.py', 'risk_aware.py', 'api_server.py']


In [2]:
from data_prep import load_data
from feature_engineering import engineer_all

raw, is_real = load_data()
df = engineer_all(raw)
print(f"Data source: {'REAL Kaggle dataset' if is_real else 'SYNTHETIC placeholder'}")
print(f"Shape: {df.shape}")
df.head()

Data source: SYNTHETIC placeholder
Shape: (1440, 20)


,ship_id,ship_type,route_id,month,distance,fuel_type,fuel_consumption,CO2_emissions,weather_conditions,engine_efficiency,implied_speed_knots,nominal_speed_knots,actual_time_hours,expected_time_hours,delay_hours,fuel_price_usd_per_litre,bunker_cost_usd,demurrage_rate_usd_per_hour,demurrage_cost_usd,total_voyage_cost_usd
0,NG000,Oil Service Boat,R9,January,302.55,Diesel,6239.16,16562.30,Moderate,73.29,8.913535,10.0,33.942761,30.255,3.687761,0.72,4492.1952,45,165.949253,4658.144453
1,NG000,Oil Service Boat,R6,February,218.15,Diesel,6364.51,17633.70,Calm,90.28,10.602063,10.0,20.576184,21.815,-1.238816,0.72,4582.4472,45,0.000000,4582.447200
2,NG000,Oil Service Boat,R5,March,125.48,Diesel,4274.32,11391.66,Moderate,79.25,11.455967,10.0,10.953244,12.548,-1.594756,0.72,3077.5104,45,0.000000,3077.510400
3,NG000,Oil Service Boat,R6,April,145.10,Diesel,3957.24,10774.75,Stormy,84.07,10.250570,10.0,14.155311,14.510,-0.354689,0.72,2849.2128,45,0.000000,2849.212800
4,NG000,Oil Service Boat,R1,May,265.40,Diesel,6018.26,15735.18,Moderate,95.00,9.346963,10.0,28.394250,26.540,1.854250,0.72,4333.1472,45,83.441241,4416.588441


In [3]:
from models import train_polynomial_regression, train_random_forest, check_lstm_feasibility, train_lstm

_, poly_metrics = train_polynomial_regression(df)
print(poly_metrics)

_, rf_metrics, importance_df = train_random_forest(df)
print(rf_metrics)
importance_df.head(8)

{'model': 'Polynomial Regression (deg=2)', 'MAE': 1980.0724182336942, 'R2': 0.8597999029600125}
{'model': 'Random Forest', 'MAE': 1253.5615958762864, 'R2': 0.9197297170145189}


,feature,importance
5,ship_type_Tanker Ship,0.545581
0,distance,0.383773
1,engine_efficiency,0.034974
3,ship_type_Oil Service Boat,0.010495
10,weather_conditions_Stormy,0.006610
8,weather_conditions_Calm,0.005909
4,ship_type_Surfer Boat,0.004267
6,fuel_type_Diesel,0.002502


In [4]:
feasible, counts = check_lstm_feasibility(df)
print(f"LSTM feasible: {feasible} (median voyages/ship: {counts.median()})")

if feasible:
    try:
        _, _, lstm_metrics = train_lstm(df)
        print(lstm_metrics)
    except ValueError as e:
        print(f"LSTM skipped: {e}")

LSTM feasible: True (median voyages/ship: 12.0)
{'model': 'LSTM (window=5)', 'MAE': 3244.497549874442, 'R2': 0.5510245314419681, 'n_sequences': 840}


In [5]:
from robustness_check import run as run_robustness

overfit_result, cv_result, error_result = run_robustness(df)

Running robustness checks on Random Forest...

Train R2: 0.9833 | Test R2: 0.9197 | Gap: 0.0636 | MODERATE gap -- some overfitting.
5-fold CV -- Mean R2: 0.9000, Std: 0.0124

By ship type:
ship_type
Fishing Trawler     23.7
Oil Service Boat    23.2
Surfer Boat         21.8
Tanker Ship         17.5

By weather:
weather_conditions
Calm        23.5
Moderate    19.9
Stormy      19.1


In [6]:
from model_tuning import tune_random_forest, predict_with_uncertainty
import pandas as pd

tuning_result = tune_random_forest(df, n_iter=30)

Best params: {'model__n_estimators': 700, 'model__min_samples_split': 10, 'model__min_samples_leaf': 4, 'model__max_features': 0.7, 'model__max_depth': 10}
Tuned Test R2: 0.9269 | Baseline Test R2: 0.9197 | Improvement: +0.0071


In [7]:
example = pd.DataFrame([{
    "distance": 250, "engine_efficiency": 85,
    "ship_type": "Tanker Ship", "fuel_type": "HFO", "weather_conditions": "Moderate",
}])
unc = predict_with_uncertainty(tuning_result["best_pipe"], example)
print(f"Point estimate: {unc['point_estimate']:.1f} litres")
print(f"90% interval: [{unc['lower_90']:.1f}, {unc['upper_90']:.1f}] litres")

Point estimate: 19446.7 litres
90% interval: [14951.5, 23191.8] litres


In [8]:
from data_prep import NOMINAL_SPEED_KNOTS
from pareto import recommend_speed

for ship_type in NOMINAL_SPEED_KNOTS:
    best, curve, comparison = recommend_speed(df, ship_type, distance=250, fuel_type="HFO", weather="Moderate")
    print(f"{ship_type}:")
    print(f"  Nominal:  {comparison['nominal_speed_knots']:.1f} kn -> ${comparison['nominal_total_cost_usd']:,.0f}")
    print(f"  Optimal:  {comparison['optimal_speed_knots']:.1f} kn -> ${comparison['optimal_total_cost_usd']:,.0f}")
    print(f"  Savings:  ${comparison['savings_usd']:,.0f} ({comparison['savings_pct']:.1f}%)\n")

Oil Service Boat:
  Nominal:  9.9 kn -> $2,913
  Optimal:  5.8 kn -> $1,805
  Savings:  $1,108 (38.0%)

Fishing Trawler:
  Nominal:  7.9 kn -> $2,343
  Optimal:  4.0 kn -> $1,217
  Savings:  $1,126 (48.1%)

Surfer Boat:
  Nominal:  19.9 kn -> $1,820
  Optimal:  9.7 kn -> $897
  Savings:  $923 (50.7%)

Tanker Ship:
  Nominal:  13.9 kn -> $8,186
  Optimal:  7.0 kn -> $4,211
  Savings:  $3,975 (48.6%)



In [9]:
from risk_aware import recommend_speed_risk_aware

for ship_type in NOMINAL_SPEED_KNOTS:
    result, curve = recommend_speed_risk_aware(df, tuning_result["best_pipe"], ship_type, distance=250, risk_aversion=1.0)
    print(f"{ship_type} (fuel-prediction uncertainty: {result['relative_uncertainty_pct']:.1f}%)")
    print(f"  Naive (risk-neutral): {result['naive_speed_knots']:.1f} kn -> ${result['naive_cost_usd']:,.0f}")
    print(f"  Risk-aware:           {result['risk_aware_speed_knots']:.1f} kn -> ${result['risk_aware_point_cost_usd']:,.0f} "
          f"(worst case: ${result['risk_aware_worst_case_cost_usd']:,.0f})")
    print(f"  {result['interpretation']}\n")

Oil Service Boat (fuel-prediction uncertainty: 15.8%)
  Naive (risk-neutral): 5.8 kn -> $1,805
  Risk-aware:           5.5 kn -> $1,811 (worst case: $1,952)
  Risk-aware recommendation shifts speed by -0.3 kn to hedge against a 15.8% typical fuel-prediction error.

Fishing Trawler (fuel-prediction uncertainty: 12.5%)
  Naive (risk-neutral): 4.0 kn -> $1,217
  Risk-aware:           3.9 kn -> $1,220 (worst case: $1,290)
  Risk-aware recommendation shifts speed by -0.1 kn to hedge against a 12.5% typical fuel-prediction error.

Surfer Boat (fuel-prediction uncertainty: 14.3%)
  Naive (risk-neutral): 9.7 kn -> $897
  Risk-aware:           9.4 kn -> $900 (worst case: $958)
  Risk-aware recommendation shifts speed by -0.3 kn to hedge against a 14.3% typical fuel-prediction error.

Tanker Ship (fuel-prediction uncertainty: 13.0%)
  Naive (risk-neutral): 7.0 kn -> $4,211
  Risk-aware:           6.8 kn -> $4,220 (worst case: $4,476)
  Risk-aware recommendation shifts speed by -0.2 kn to hedge a

In [10]:
!pip install -q flask flask-cors

from api_server import create_app
import threading, time

tuned = tuning_result["best_pipe"]
app = create_app(df, tuned_pipe=tuned, is_real=is_real)

def run_api():
    app.run(port=5000)

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
time.sleep(2)
print("Live API server running on port 5000 inside this Colab session.")

 * Serving Flask app 'api_server'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


Live API server running on port 5000 inside this Colab session.


In [11]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess, re

process = subprocess.Popen(
    ['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

live_api_url = None
for line in process.stdout:
    print(line, end='')
    match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if match:
        live_api_url = match.group(0)
        print("\n" + "=" * 60)
        print("PASTE THIS URL INTO THE WEBSITE, THEN CLICK CONNECT:")
        print(live_api_url)
        print("=" * 60)
        break

2026-09-17T01:45:06Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-17T01:45:06Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-17T01:45:09Z INF +--------------------------------------------------------------------------------------------+
2026-09-17T01:45:09Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-17T01:45:09Z INF |  https://nevada-rare-joan-fundamental.trycloudflare.co

In [12]:
with open('voyage_speed_optimizer.html', 'w') as f:
    f.write("<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n<meta charset=\"UTF-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">\n<title>Voyage Speed Optimizer \u2014 Colab Display</title>\n<script src=\"https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.0/chart.umd.min.js\"></script>\n<script src=\"https://cdnjs.cloudflare.com/ajax/libs/highlight.js/11.9.0/highlight.min.js\"></script>\n<link rel=\"stylesheet\" href=\"https://cdnjs.cloudflare.com/ajax/libs/highlight.js/11.9.0/styles/atom-one-dark.min.css\">\n<style>\n  :root {\n    --navy: #0B2545; --navy-light: #13335E; --teal: #1B7A8C; --teal-light: #2C8FA3;\n    --amber: #F2A93B; --bg: #F7F9FC; --panel: #FFFFFF; --ink: #0F172A;\n    --muted: #64748B; --border: #E2E8F0; --danger: #C0392B; --success: #16A34A;\n  }\n  * { box-sizing: border-box; }\n  body { margin: 0; font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: var(--bg); color: var(--ink); line-height: 1.5; }\n  header { background: var(--navy); color: white; padding: 24px 32px; }\n  header h1 { margin: 0 0 6px 0; font-size: 22px; font-weight: 700; }\n  header p { margin: 0; color: #9FB8D1; font-size: 13.5px; max-width: 700px; }\n\n  .dependency-banner {\n    background: #FFF7E6; border-bottom: 2px solid var(--amber);\n    padding: 10px 32px; font-size: 12.5px; color: #92600A; font-weight: 600;\n  }\n\n  .layout { max-width: 1100px; margin: 0 auto; padding: 28px 24px; }\n\n  .connect-panel {\n    background: var(--panel); border: 1.5px solid var(--border); border-radius: 10px;\n    padding: 22px; margin-bottom: 24px;\n  }\n  .connect-panel h2 { margin: 0 0 6px 0; font-size: 16px; color: var(--navy); }\n  .connect-panel p { margin: 0 0 14px 0; font-size: 13px; color: var(--muted); }\n  .connect-row { display: flex; gap: 10px; }\n  .connect-row input {\n    flex: 1; padding: 10px 12px; border: 1px solid var(--border); border-radius: 6px;\n    font-size: 13px; font-family: 'Consolas', monospace;\n  }\n  .btn {\n    background: var(--teal); color: white; border: none; border-radius: 6px;\n    padding: 10px 20px; font-size: 13.5px; font-weight: 600; cursor: pointer; font-family: inherit;\n    white-space: nowrap;\n  }\n  .btn:hover { background: var(--teal-light); }\n  .btn:disabled { background: var(--border); color: var(--muted); cursor: not-allowed; }\n\n  .status-badge {\n    display: inline-flex; align-items: center; gap: 6px; margin-top: 12px;\n    font-size: 12.5px; font-weight: 600; padding: 6px 12px; border-radius: 20px;\n  }\n  .status-badge.disconnected { background: #FEF2F2; color: var(--danger); }\n  .status-badge.connected { background: #F0FDF4; color: var(--success); }\n  .status-dot { width: 8px; height: 8px; border-radius: 50%; background: currentColor; }\n\n  .main-panel {\n    background: var(--panel); border: 1.5px solid var(--border); border-radius: 10px;\n    padding: 24px; position: relative;\n  }\n  .main-panel.inert::before {\n    content: \"Waiting for Colab connection \u2014 controls are disabled until connected above.\";\n    position: absolute; inset: 0; background: rgba(247,249,252,0.92);\n    display: flex; align-items: center; justify-content: center; text-align: center;\n    padding: 40px; font-size: 15px; font-weight: 600; color: var(--muted);\n    border-radius: 10px; z-index: 5;\n  }\n\n  .controls-row { display: grid; grid-template-columns: repeat(5, 1fr); gap: 14px; margin-bottom: 24px; }\n  @media (max-width: 900px) { .controls-row { grid-template-columns: 1fr 1fr; } }\n  .control-group label { display: block; font-size: 12px; font-weight: 600; color: var(--navy); margin-bottom: 6px; }\n  select { width: 100%; padding: 9px 10px; border: 1px solid var(--border); border-radius: 6px; font-size: 13.5px; font-family: inherit; background: white; }\n  input[type=\"range\"] { width: 100%; accent-color: var(--teal); }\n  .range-value { font-size: 12px; color: var(--teal); font-weight: 600; }\n\n  .result-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 14px; margin-bottom: 20px; }\n  @media (max-width: 700px) { .result-grid { grid-template-columns: 1fr; } }\n  .result-card { background: var(--bg); border: 1px solid var(--border); border-radius: 8px; padding: 16px; }\n  .result-card.highlight { border: 1.5px solid var(--amber); background: #FFFBF3; }\n  .result-label { font-size: 11px; color: var(--muted); font-weight: 600; margin-bottom: 8px; }\n  .result-value { font-size: 22px; font-weight: 700; color: var(--navy); }\n  .result-detail { font-size: 12px; color: var(--muted); margin-top: 4px; }\n\n  .chart-container { height: 280px; position: relative; margin-top: 10px; }\n\n  .footer-note {\n    background: #F1F5F9; border-radius: 8px; padding: 14px 16px; margin-top: 20px;\n    font-size: 12px; color: var(--muted);\n  }\n\n  .tabs { display: flex; gap: 6px; margin: 24px 0 12px 0; }\n  .tab-btn { background: var(--panel); border: 1px solid var(--border); border-radius: 6px; padding: 8px 14px; font-size: 12.5px; font-weight: 600; color: var(--muted); cursor: pointer; font-family: inherit; }\n  .tab-btn.active { background: var(--navy); color: white; border-color: var(--navy); }\n  pre { border-radius: 8px; padding: 18px !important; font-size: 12px; max-height: 500px; overflow: auto; margin: 0; }\n</style>\n</head>\n<body>\n\n<header>\n  <h1>Voyage Speed Optimizer</h1>\n  <p>This page performs no calculations of its own. Every number and chart below is computed live by the Colab notebook and sent here on request.</p>\n</header>\n<div class=\"dependency-banner\">\n  This display does nothing without an active Colab connection. It cannot calculate, estimate, or simulate anything independently.\n</div>\n\n<div class=\"layout\">\n\n  <div class=\"connect-panel\">\n    <h2>Connect to Colab</h2>\n    <p>Run the notebook through the API + tunnel cells, then paste the printed URL here.</p>\n    <div class=\"connect-row\">\n      <input type=\"text\" id=\"apiUrl\" placeholder=\"https://xxxx.trycloudflare.com\">\n      <button class=\"btn\" id=\"connectBtn\">Connect</button>\n    </div>\n    <div class=\"status-badge disconnected\" id=\"statusBadge\">\n      <span class=\"status-dot\"></span>\n      <span id=\"statusText\">Not connected</span>\n    </div>\n  </div>\n\n  <div class=\"main-panel inert\" id=\"mainPanel\">\n    <div class=\"controls-row\">\n      <div class=\"control-group\">\n        <label for=\"shipType\">Ship type</label>\n        <select id=\"shipType\" disabled>\n          <option>Oil Service Boat</option>\n          <option>Fishing Trawler</option>\n          <option>Surfer Boat</option>\n          <option selected>Tanker Ship</option>\n        </select>\n      </div>\n      <div class=\"control-group\">\n        <label for=\"distance\">Distance: <span id=\"distanceValue\" class=\"range-value\">250</span> nm</label>\n        <input type=\"range\" id=\"distance\" min=\"20\" max=\"500\" step=\"10\" value=\"250\" disabled>\n      </div>\n      <div class=\"control-group\">\n        <label for=\"weather\">Weather</label>\n        <select id=\"weather\" disabled>\n          <option>Calm</option>\n          <option selected>Moderate</option>\n          <option>Stormy</option>\n        </select>\n      </div>\n      <div class=\"control-group\">\n        <label for=\"fuelType\">Fuel type</label>\n        <select id=\"fuelType\" disabled>\n          <option selected>HFO</option>\n          <option>Diesel</option>\n        </select>\n      </div>\n      <div class=\"control-group\">\n        <label for=\"maxDelay\">Max delay: <span id=\"maxDelayValue\" class=\"range-value\">10</span> hrs</label>\n        <input type=\"range\" id=\"maxDelay\" min=\"0\" max=\"60\" step=\"1\" value=\"10\" disabled>\n      </div>\n    </div>\n\n    <div class=\"result-grid\">\n      <div class=\"result-card\">\n        <div class=\"result-label\">Normal speed (no optimization)</div>\n        <div class=\"result-value\" id=\"normalCost\">\u2014</div>\n        <div class=\"result-detail\" id=\"normalDetail\">Waiting for Colab...</div>\n      </div>\n      <div class=\"result-card\">\n        <div class=\"result-label\">Cheapest option</div>\n        <div class=\"result-value\" id=\"cheapestCost\">\u2014</div>\n        <div class=\"result-detail\" id=\"cheapestDetail\">Waiting for Colab...</div>\n      </div>\n      <div class=\"result-card highlight\">\n        <div class=\"result-label\">Your choice (respects delay limit)</div>\n        <div class=\"result-value\" id=\"choiceCost\">\u2014</div>\n        <div class=\"result-detail\" id=\"choiceDetail\">Waiting for Colab...</div>\n      </div>\n    </div>\n\n    <div class=\"chart-container\"><canvas id=\"liveChart\"></canvas></div>\n\n    <div class=\"footer-note\" id=\"footerNote\">No data received yet \u2014 connect to Colab above.</div>\n  </div>\n\n  <div class=\"tabs\">\n    <button class=\"tab-btn active\" data-tab=\"live\">Live Display</button>\n    <button class=\"tab-btn\" data-tab=\"code\">Colab Source (reference only)</button>\n  </div>\n\n  <div id=\"tab-code\" style=\"display:none;\">\n    <div class=\"main-panel\" style=\"padding-top:16px;\">\n      <p style=\"font-size:13px; color:var(--muted); margin-top:0;\">\n        This is the actual Python running inside Colab that produces every number above.\n        It is shown here for transparency only \u2014 this code does not run in your browser.\n      </p>\n      <pre><code class=\"language-python\" id=\"codeDisplay\"></code></pre>\n    </div>\n  </div>\n\n</div>\n\n<script>\n// ============================================================\n// This file contains NO fuel/cost/speed calculation logic.\n// It only sends requests to a live Colab session and displays\n// whatever comes back. If apiBase is null, nothing here works.\n// ============================================================\nlet apiBase = null;\nlet liveChart = null;\n\nconst statusBadge = document.getElementById('statusBadge');\nconst statusText = document.getElementById('statusText');\nconst mainPanel = document.getElementById('mainPanel');\nconst controlIds = ['shipType', 'distance', 'weather', 'fuelType', 'maxDelay'];\n\nfunction setConnected(isConnected, message) {\n  statusBadge.classList.toggle('connected', isConnected);\n  statusBadge.classList.toggle('disconnected', !isConnected);\n  statusText.textContent = message;\n  mainPanel.classList.toggle('inert', !isConnected);\n  controlIds.forEach(id => document.getElementById(id).disabled = !isConnected);\n}\n\ndocument.getElementById('connectBtn').addEventListener('click', async function() {\n  let url = document.getElementById('apiUrl').value.trim();\n  if (url.endsWith('/')) url = url.slice(0, -1);\n  if (!url) { setConnected(false, 'Enter a URL first.'); return; }\n\n  setConnected(false, 'Connecting...');\n  try {\n    const res = await fetch(url + '/health');\n    if (!res.ok) throw new Error('HTTP ' + res.status);\n    const data = await res.json();\n    apiBase = url;\n    setConnected(true, `Connected \u2014 ${data.data_source}, ${data.rows_loaded} rows (live in Colab right now)`);\n    fetchAndDisplay();\n  } catch (err) {\n    apiBase = null;\n    setConnected(false, 'Could not reach Colab: ' + err.message);\n  }\n});\n\nasync function fetchAndDisplay() {\n  if (!apiBase) return;\n\n  const shipType = document.getElementById('shipType').value;\n  const distance = parseFloat(document.getElementById('distance').value);\n  const weather = document.getElementById('weather').value;\n  const fuelType = document.getElementById('fuelType').value;\n  const maxDelay = parseFloat(document.getElementById('maxDelay').value);\n\n  document.getElementById('distanceValue').textContent = distance;\n  document.getElementById('maxDelayValue').textContent = maxDelay;\n\n  document.getElementById('footerNote').textContent = 'Asking Colab to compute this...';\n\n  try {\n    const res = await fetch(apiBase + '/recommend', {\n      method: 'POST',\n      headers: { 'Content-Type': 'application/json' },\n      body: JSON.stringify({ ship_type: shipType, distance, weather, fuel_type: fuelType, max_delay_hours: maxDelay }),\n    });\n    if (!res.ok) throw new Error('HTTP ' + res.status);\n    const data = await res.json();\n\n    document.getElementById('normalCost').textContent = '$' + data.nominal.cost_usd.toLocaleString();\n    document.getElementById('normalDetail').textContent = data.nominal.speed_knots + ' kn';\n\n    document.getElementById('cheapestCost').textContent = '$' + data.cheapest.cost_usd.toLocaleString();\n    document.getElementById('cheapestDetail').textContent = data.cheapest.speed_knots + ' kn \u2014 saves ' + data.cheapest.savings_pct + '%';\n\n    document.getElementById('choiceCost').textContent = '$' + data.your_choice.cost_usd.toLocaleString();\n    document.getElementById('choiceDetail').textContent = data.your_choice.speed_knots + ' kn \u2014 delay ' + data.your_choice.delay_hours + ' hrs';\n\n    document.getElementById('footerNote').textContent =\n      'Computed live by Colab just now \u2014 ' + data.data_source + ', ' + data.rows_used + ' rows.';\n\n    const ctx = document.getElementById('liveChart');\n    if (liveChart) liveChart.destroy();\n    liveChart = new Chart(ctx, {\n      type: 'line',\n      data: {\n        labels: data.speed_curve.speed_knots.map(s => s.toFixed(1)),\n        datasets: [{ label: 'Total cost (from Colab)', data: data.speed_curve.total_cost_usd,\n          borderColor: '#0B2545', backgroundColor: 'rgba(11,37,69,0.06)', borderWidth: 2, pointRadius: 0, fill: true }]\n      },\n      options: {\n        responsive: true, maintainAspectRatio: false,\n        plugins: { legend: { display: false } },\n        scales: { x: { title: { display: true, text: 'Speed (knots)' } }, y: { title: { display: true, text: 'Cost (USD)' } } }\n      }\n    });\n  } catch (err) {\n    document.getElementById('footerNote').textContent = 'Lost connection to Colab: ' + err.message;\n    setConnected(false, 'Disconnected \u2014 ' + err.message);\n  }\n}\n\ncontrolIds.forEach(id => {\n  document.getElementById(id).addEventListener('input', fetchAndDisplay);\n});\n\ndocument.querySelectorAll('.tab-btn').forEach(btn => {\n  btn.addEventListener('click', () => {\n    document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));\n    btn.classList.add('active');\n    document.getElementById('tab-live').style.display = btn.dataset.tab === 'live' ? 'block' : 'none';\n    document.getElementById('tab-code').style.display = btn.dataset.tab === 'code' ? 'block' : 'none';\n  });\n});\n\nconst PYTHON_SOURCE = \"# api_server.py -- this is what actually computes everything shown above\\\\n\\\\nfrom pareto import recommend_speed, recommend_speed_with_max_delay\\\\n\\\\n@app.route('/recommend', methods=['POST'])\\\\ndef recommend():\\\\n    data = request.get_json()\\\\n    ship_type = data.get('ship_type')\\\\n    distance = float(data.get('distance'))\\\\n    weather = data.get('weather')\\\\n    fuel_type = data.get('fuel_type')\\\\n    max_delay = float(data.get('max_delay_hours'))\\\\n\\\\n    # df is the REAL dataset loaded in this Colab session\\\\n    best, curve, comparison = recommend_speed(df, ship_type, distance, fuel_type, weather)\\\\n    constrained, _ = recommend_speed_with_max_delay(df, ship_type, distance, max_delay, fuel_type, weather)\\\\n\\\\n    return jsonify({\\\\n        'nominal': {'speed_knots': comparison['nominal_speed_knots']},\\\\n        'cheapest': {'speed_knots': comparison['optimal_speed_knots']},\\\\n        'your_choice': {'speed_knots': constrained['constrained_speed_knots']},\\\\n        'speed_curve': curve[['speed_knots', 'total_cost_usd']].to_dict(orient='list'),\\\\n    })\\\\n\\\\n# The website's JavaScript only calls this endpoint and displays the\\\\n# response -- it contains no fuel, speed, or cost formulas of its own.\";\n\ndocument.getElementById('codeDisplay').textContent = PYTHON_SOURCE;\nhljs.highlightElement(document.getElementById('codeDisplay'));\n</script>\n\n</body>\n</html>\n")

import http.server, socketserver, threading as th, time
from google.colab import output

PORT = 8000
socketserver.TCPServer.allow_reuse_address = True

class Handler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        pass

def start_server():
    try:
        with socketserver.TCPServer(("", PORT), Handler) as httpd:
            httpd.serve_forever()
    except OSError:
        pass

th.Thread(target=start_server, daemon=True).start()
time.sleep(1)

print("Opening the website below -- paste the URL from above, click Connect.")
print("Until you connect, every control is disabled and every value shows '-'.")
output.serve_kernel_port_as_iframe(PORT, path='/voyage_speed_optimizer.html', height=900)

Opening the website below -- paste the URL from above, click Connect.
Until you connect, every control is disabled and every value shows '-'.


<IPython.core.display.Javascript object>